# Cannibalization: Naive vs. Fixed

## The Pair
- **Promoted Product (A):** `1127831` (Strawberries, National brand, 16 OZ)
- **Neighbor Product (B):** `860299` (Strawberries, National brand, 16 OZ)

**Why this pair?** Same sub-commodity, same brand tier. A plausible substitution story.

In [1]:
# Setup connection for the whole notebook
import duckdb
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

con = duckdb.connect("../data/processed/retail.duckdb", read_only=True)
print("Database connected.")

Database connected.


In [2]:
# The Naive Comparison Query (Day 19)
con.sql("""
    WITH promo_weeks AS (
        SELECT DISTINCT week_no
        FROM product_store_week
        WHERE product_id = 1127831 AND is_price_promo AND NOT is_featured
    ),
    a_weekly AS (
        SELECT week_no, SUM(total_quantity) AS qty
        FROM product_store_week WHERE product_id = 1127831
        GROUP BY week_no
    ),
    b_weekly AS (
        SELECT week_no, SUM(total_quantity) AS qty
        FROM product_store_week WHERE product_id = 860299
        GROUP BY week_no
    )
    SELECT
        'A (1127831)' AS product,
        SUM(CASE WHEN p.week_no IS NOT NULL THEN qty ELSE 0 END) AS total_units_during,
        SUM(CASE WHEN p.week_no IS NULL THEN qty ELSE 0 END) AS total_units_baseline,
        COUNT(CASE WHEN p.week_no IS NOT NULL THEN 1 END) AS n_weeks_during,
        COUNT(CASE WHEN p.week_no IS NULL THEN 1 END) AS n_weeks_baseline,
        ROUND( (SUM(CASE WHEN p.week_no IS NOT NULL THEN qty ELSE 0 END) / NULLIF(COUNT(CASE WHEN p.week_no IS NOT NULL THEN 1 END), 0)) / 
               (SUM(CASE WHEN p.week_no IS NULL THEN qty ELSE 0 END) / NULLIF(COUNT(CASE WHEN p.week_no IS NULL THEN 1 END), 0)) - 1, 4) AS pct_change
    FROM a_weekly a
    LEFT JOIN promo_weeks p ON a.week_no = p.week_no

    UNION ALL

    SELECT
        'B (860299)' AS product,
        SUM(CASE WHEN p.week_no IS NOT NULL THEN qty ELSE 0 END) AS total_units_during,
        SUM(CASE WHEN p.week_no IS NULL THEN qty ELSE 0 END) AS total_units_baseline,
        COUNT(CASE WHEN p.week_no IS NOT NULL THEN 1 END) AS n_weeks_during,
        COUNT(CASE WHEN p.week_no IS NULL THEN 1 END) AS n_weeks_baseline,
        ROUND( (SUM(CASE WHEN p.week_no IS NOT NULL THEN qty ELSE 0 END) / NULLIF(COUNT(CASE WHEN p.week_no IS NOT NULL THEN 1 END), 0)) / 
               (SUM(CASE WHEN p.week_no IS NULL THEN qty ELSE 0 END) / NULLIF(COUNT(CASE WHEN p.week_no IS NULL THEN 1 END), 0)) - 1, 4) AS pct_change
    FROM b_weekly b
    LEFT JOIN promo_weeks p ON b.week_no = p.week_no
""").show()

┌─────────────┬────────────────────┬──────────────────────┬────────────────┬──────────────────┬────────────┐
│   product   │ total_units_during │ total_units_baseline │ n_weeks_during │ n_weeks_baseline │ pct_change │
│   varchar   │       int128       │        int128        │     int64      │      int64       │   double   │
├─────────────┼────────────────────┼──────────────────────┼────────────────┼──────────────────┼────────────┤
│ A (1127831) │               2752 │                 4760 │             24 │               70 │     0.6863 │
│ B (860299)  │                268 │                  470 │             15 │               23 │    -0.1257 │
└─────────────┴────────────────────┴──────────────────────┴────────────────┴──────────────────┴────────────┘



## The Naive Result (Day 19)
Comparing A's promo weeks vs. B's baseline weeks.

| Product | Avg Units During Promo | Avg Units Baseline | Naive % Change |
|---|---:|---:|---:|
| **A (1127831)** | 114.7 | 68.0 | **+68.6%** |
| **B (860299)** | 17.9 | 20.4 | **-12.6%** |

*Interpretation delayed. The goal is to break this naive number.*

## The DiD Failure (Day 20)

We attempted a Difference-in-Differences (DiD) fix using Product C (1135476) as a control. 

1. **The INNER JOIN bug:** The initial DiD query used an `INNER JOIN` between B and C. Because C almost never sold, the join dropped 21 of A's 24 promo weeks, inflating B's average from 17.9 to 77.0. (Corrected numbers: B: 17.9 during, 20.4 baseline).
2. **The Category-wide check:** We checked the promo rate of the whole store during A's promo weeks. Overall baseline: 13.41%. During A's promo weeks: 13.71%. The whole store was *not* on sale, so this is a category-specific effect.
3. **The DiD impossibility:** Because the entire Strawberry category is promoted in lockstep campaign bursts, a "clean" comparison of non-promoted weeks is impossible.

## The Fix: Continuous Depth Regression (Day 21)

Binary promo flags fail because the whole category gets promoted at once. We pivot to a regression using **continuous discount depth**.

**The Model:** `log(B_quantity) ~ A_discount_depth + B_discount_depth + C(week_no)`

- `A_discount_depth`: Captures the cannibalization effect (does a deeper discount on A steal more from B?).
- `B_discount_depth`: Controls for B's own promotions (the baseline contamination).
- `C(week_no)`: Week fixed effects, absorbing the campaign clustering.

In [1]:
import duckdb
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import warnings
from statsmodels.tools.sm_exceptions import SingularMatrixWarning

con = duckdb.connect("../data/processed/retail.duckdb", read_only=True)

# FIX Blocker 1 & 2:
# Removed the is_price_promo filter on A so depth_A can be 0 (baseline).
# Used INNER JOIN to restrict to stores that actually carry B.
query = """
    WITH a_data AS (
        SELECT store_id, week_no, total_quantity AS qty_A, discount_depth AS depth_A
        FROM product_store_week
        WHERE product_id = 1127831
    ),
    b_data AS (
        SELECT store_id, week_no, total_quantity AS qty_B, discount_depth AS depth_B
        FROM product_store_week
        WHERE product_id = 860299
    )
    SELECT 
        a.store_id,
        a.week_no,
        a.qty_A,
        a.depth_A,
        COALESCE(b.qty_B, 0) AS qty_B,
        COALESCE(b.depth_B, 0) AS depth_B
    FROM a_data a
    -- INNER JOIN to only include stores where B is actually carried
    INNER JOIN (SELECT DISTINCT store_id FROM b_data) b_stores 
        ON a.store_id = b_stores.store_id
    LEFT JOIN b_data b 
        ON a.store_id = b.store_id 
       AND a.week_no = b.week_no
"""
df = con.sql(query).df()
df['log_B_qty'] = np.log(df['qty_B'] + 1)

print(f"Total observations: {len(df)}")

# FIX Blocker 3: Wrap in SingularMatrixWarning check
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    model = smf.ols("log_B_qty ~ depth_A + depth_B + C(week_no)", data=df).fit(
        cov_type="cluster", cov_kwds={"groups": df["store_id"]}
    )
    singular = any(issubclass(w.category, SingularMatrixWarning) for w in caught)

if singular:
    print("WARNING: Singular matrix detected! The coefficients are unreliable.")
else:
    print("\n--- Regression Summary ---")
    print(f"Coefficient for A's discount depth: {model.params['depth_A']:.3f}")
    print(f"P-value: {model.pvalues['depth_A']:.4f}")
    print(f"95% CI: {model.conf_int().loc['depth_A'].tolist()}")
    print(f"df_resid: {model.df_resid:.0f}")


Total observations: 3199

--- Regression Summary ---
Coefficient for A's discount depth: -0.047
P-value: 0.2556
95% CI: [-0.12913722683763582, 0.03432827116789869]
df_resid: 3103


## Regression Results & Verdict (Day 21)

After correcting the panel to include non-promo weeks (restoring the baseline) and restricting to stores that carry both products, we ran the continuous depth regression with week fixed effects.

- **Total Observations:** 3,199
- **Coefficient for A's discount depth:** -0.047
- **P-value:** 0.2556
- **95% CI:** [-0.129, 0.034]

**Interpretation:**
The coefficient is essentially zero and statistically insignificant (CI crosses zero). We cannot detect cannibalization for this pair. Even when A goes on a deep discount, B's sales do not meaningfully drop.

**The Precision Caveat:**
The 0.887 correlation between A's and B's depths (due to category-wide campaigns) mathematically inflates our standard errors, making the CI wide. However, even with this limitation, the point estimate is so close to zero that the honest conclusion is a null result: no measurable cannibalization.

In [3]:
con.sql("""
    WITH a_data AS (
        SELECT store_id, week_no, discount_depth AS depth_A
        FROM product_store_week
        WHERE product_id = 1127831
    ),
    b_data AS (
        SELECT store_id, week_no, discount_depth AS depth_B
        FROM product_store_week
        WHERE product_id = 860299
    )
    SELECT 
        CORR(a.depth_A, COALESCE(b.depth_B, 0)) AS regression_panel_corr,
        COUNT(*) AS n_rows
    FROM a_data a
    INNER JOIN (SELECT DISTINCT store_id FROM b_data) b_stores 
        ON a.store_id = b_stores.store_id
    LEFT JOIN b_data b 
        ON a.store_id = b.store_id 
       AND a.week_no = b.week_no
""").show()

┌───────────────────────┬────────┐
│ regression_panel_corr │ n_rows │
│        double         │ int64  │
├───────────────────────┼────────┤
│   0.17635219270573793 │   3199 │
└───────────────────────┴────────┘



**The Precision Caveat:**
While the initial promo-only data showed an 0.887 correlation (perfect lockstep), the full 3,199-row regression panel has a correlation of just **0.176**. The inclusion of non-promo weeks introduces independent variation. Therefore, the wide confidence interval [-0.129, 0.034] is not caused by collinearity; it simply reflects that the true effect is near zero and the observational data is noisy. The null result is robust.